# Simple Kaggle Inference

Versi ini tetap untuk Kaggle: install dependency, download model dari Hugging Face, load policy, lalu inference 1 gambar.

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

REPO_ID = "izunx/drone-roblok-2500"
BASE_DIR = Path("/kaggle/working/dronepivla")
OPENPI_DIR = BASE_DIR / "openpi"
MODEL_DIR = BASE_DIR

IMAGE_PATH = None  # contoh: "/kaggle/input/my-dataset/frame.jpg"
TASK = "look at the red cube"

if not BASE_DIR.exists():
    snapshot_download(
        repo_id=REPO_ID,
        local_dir=str(BASE_DIR),
        local_dir_use_symlinks=False,
    )

required_files = [
    MODEL_DIR / "model.safetensors",
    MODEL_DIR / "norm_stats.json",
    MODEL_DIR / "metadata.pt",
    OPENPI_DIR,
]

for path in required_files:
    print(f"{'OK':>2} | {path}" if path.exists() else f"NO | {path}")

In [ ]:
import subprocess
import sys


def pip_install(*packages: str) -> None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-warn-conflicts",
        *packages,
    ])


subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "jax", "jaxlib", "numpy", "torch", "torchaudio", "torchvision"],
    check=False,
    capture_output=True,
)

pip_install("numpy==1.26.4")
pip_install(
    "torch==2.4.0",
    "torchvision",
    "torchaudio",
    "--index-url",
    "https://download.pytorch.org/whl/cu121",
)
pip_install(
    "jax[cuda12]==0.5.3",
    "jaxlib==0.5.3",
    "flax==0.10.2",
    "orbax-checkpoint==0.11.13",
    "chex",
    "equinox>=0.11.8",
    "ml-collections==1.0.0",
    "jaxtyping==0.2.36",
    "ml-dtypes==0.4.1",
)
pip_install(
    "safetensors>=0.4.0",
    "einops>=0.8.0",
    "wandb>=0.19.1",
    "datasets",
    "huggingface-hub",
    "peft",
    "tyro>=0.9.5",
    "beartype==0.19.0",
    "numpydantic>=1.6.6",
    "sentencepiece>=0.2.0",
    "tqdm-loggable>=0.2",
    "humanize",
    "filelock>=3.16.1",
    "treescope>=0.1.7",
    "augmax>=0.3.4",
    "dm-tree>=0.1.8",
    "flatbuffers>=24.3.25",
    "fsspec==2023.10.0",
    "gcsfs",
    "imageio>=2.36.1",
    "opencv-python-headless",
    "pillow>=11.0.0",
    "rich>=14.0.0",
    "polars>=1.30.0",
    "transformers==4.53.2",
    "gym-aloha>=0.1.1",
    "lerobot@git+https://github.com/huggingface/lerobot@0cf864870cf29f4738d3ade893e6fd13fbd7cdb5",
)
pip_install("-e", str(OPENPI_DIR), "--no-deps")

print("Dependency install selesai.")

In [ ]:
import dataclasses
import importlib
import os
import re
import shutil
import sys
import urllib.request
from pathlib import Path

OPENPI_SRC = OPENPI_DIR / "src"
OPENPI_CLIENT_SRC = OPENPI_DIR / "packages" / "openpi-client" / "src"
TRANSFORMERS_PATCH_ROOT = OPENPI_SRC / "openpi" / "models_pytorch" / "transformers_replace"
OPENPI_DOWNLOAD_PY = OPENPI_SRC / "openpi" / "shared" / "download.py"
OPENPI_MODEL_PY = OPENPI_SRC / "openpi" / "models" / "model.py"
ASSET_DIR = MODEL_DIR / "assets" / "rafli" / "drone_roblox"
TOKENIZER_PATH = Path("/root/.cache/openpi/big_vision/paligemma_tokenizer.model")

os.environ["PYTHONPATH"] = f"{OPENPI_SRC}:{OPENPI_CLIENT_SRC}"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for path in (OPENPI_SRC, OPENPI_CLIENT_SRC):
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

ASSET_DIR.mkdir(parents=True, exist_ok=True)
for name in ("norm_stats.json", "metadata.pt"):
    src = MODEL_DIR / name
    dst = ASSET_DIR / name
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

if OPENPI_DOWNLOAD_PY.exists():
    text = OPENPI_DOWNLOAD_PY.read_text()
    OPENPI_DOWNLOAD_PY.write_text(text.replace("datetime.UTC", "datetime.timezone.utc"))

# Patch loader agar checkpoint LoRA dimuat seperti notebook lama (non-strict).
if OPENPI_MODEL_PY.exists():
    model_text = OPENPI_MODEL_PY.read_text()
    lora_loader_patch = """        # --- PATCH LORA INFERENCE (KAGGLE) ---
        from safetensors.torch import load_file
        from peft import LoraConfig, get_peft_model

        lora_config = LoraConfig(
            r=8, lora_alpha=16, target_modules=\"all-linear\", bias=\"none\"
        )
        model = get_peft_model(model, lora_config)
        state_dict = load_file(weight_path)
        model.load_state_dict(state_dict, strict=False)
        # -------------------------------------
"""

    # Selalu ganti jika baris load_model lama masih ada (walau marker patch sudah pernah tersisip).
    new_text, n_subs = re.subn(
        r"^\s*safetensors\.torch\.load_model\(model,\s*weight_path\)\s*$",
        lora_loader_patch,
        model_text,
        count=1,
        flags=re.MULTILINE,
    )

    if n_subs == 1:
        OPENPI_MODEL_PY.write_text(new_text)
        print("Model loader patched to LoRA non-strict mode.")
    elif "PATCH LORA INFERENCE (KAGGLE)" in model_text:
        print("Model loader sudah ter-patch sebelumnya.")
    else:
        raise RuntimeError(
            "Gagal patch model.py: baris load_model tidak ditemukan dan marker patch juga tidak ada."
        )

transformers = importlib.import_module("transformers")
transformers_root = Path(transformers.__file__).resolve().parent
for item in TRANSFORMERS_PATCH_ROOT.iterdir():
    target = transformers_root / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, target)

TOKENIZER_PATH.parent.mkdir(parents=True, exist_ok=True)
if not TOKENIZER_PATH.exists():
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/big_vision/paligemma_tokenizer.model",
        TOKENIZER_PATH,
    )

print(f"Asset dir : {ASSET_DIR}")
print(f"Tokenizer : {TOKENIZER_PATH}")
print(f"Model patch: {OPENPI_MODEL_PY}")

In [ ]:
import dataclasses
import torch
import torch._dynamo

from openpi.policies import policy_config as _policy_config
from openpi.training import config as _config

torch._dynamo.config.suppress_errors = True
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_config = _config.get_config("pi0_drone_lite")
train_config = dataclasses.replace(
    train_config,
    model=dataclasses.replace(train_config.model, pytorch_compile_mode=None),
)

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f'Config     : {train_config.name}')
print(f'Checkpoint : {MODEL_DIR}')
print(f'Device     : {device}')

policy = _policy_config.create_trained_policy(
    train_config,
    MODEL_DIR,
    pytorch_device=device,
)

print("Policy ready")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image


def load_image(image_path: str | None) -> np.ndarray:
    if not image_path:
        image = np.zeros((224, 224, 3), dtype=np.uint8)
        image[72:152, 72:152] = [220, 50, 50]
        return image
    return np.asarray(Image.open(image_path).convert("RGB").resize((224, 224)), dtype=np.uint8)


image = load_image(IMAGE_PATH)
observation = {
    "image": image,
    "state": np.zeros(4, dtype=np.float32),
    "task": TASK,
}

plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.title(TASK)
plt.axis("off")
plt.show()

In [ ]:
with torch.no_grad():
    result = policy.infer(observation)

actions = np.asarray(result["actions"], dtype=np.float32)
print("actions.shape:", actions.shape)
print("first_action :", np.round(actions[0], 4))
actions

## Public API Setup (Kaggle)

Jalankan cell di bawah untuk start Gradio backend dengan **public URL**:
- Akan generate link publik (e.g., `https://xxxxx-gradio.hf.space`)
- Bisa diakses dari mana saja (phone, laptop, server lain, dll)
- Support POST API requests
- Temporary (hilang setelah notebook tutup)

In [ ]:
# Setup Gradio Backend dengan Public API
# Bisa di-access dari mana saja (public IP/URL)
import gradio as gr

print("Setting up Gradio backend with public API...")

def infer_api(image_input, task_input):
    """Inference API untuk POST requests."""
    if image_input is None:
        return {"error": "Image required"}
    
    if not task_input or not task_input.strip():
        return {"error": "Task description required"}
    
    # Ensure image is numpy (224, 224, 3)
    if isinstance(image_input, Image.Image):
        image = np.asarray(
            image_input.convert("RGB").resize((224, 224)),
            dtype=np.uint8
        )
    else:
        image = np.asarray(image_input, dtype=np.uint8)
        if image.shape != (224, 224, 3):
            image = Image.fromarray(image).resize((224, 224))
            image = np.asarray(image, dtype=np.uint8)
    
    # Prepare observation
    observation = {
        "image": image,
        "state": np.zeros(4, dtype=np.float32),
        "task": task_input.strip(),
    }
    
    # Run inference
    with torch.no_grad():
        result = policy.infer(observation)
    
    actions = np.asarray(result["actions"], dtype=np.float32)
    
    return {
        "actions": actions.tolist(),
        "shape": list(actions.shape),
        "first_action": np.round(actions[0], 4).tolist(),
    }


# Create Gradio interface
with gr.Blocks(title="Pi0 Drone VLA - Kaggle") as demo:
    gr.Markdown("# Pi0 Drone VLA Inference (Public API)")
    gr.Markdown("Akses dari mana saja menggunakan public Gradio URL")
    
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(label="Input Image", type="pil")
            task_input = gr.Textbox(
                label="Task Description",
                placeholder="e.g., look at the red cube",
                lines=2,
            )
            submit_btn = gr.Button("Infer", variant="primary")
        
        with gr.Column():
            output_json = gr.JSON(label="Result (actions, shape, first_action)")
    
    submit_btn.click(
        fn=infer_api,
        inputs=[image_input, task_input],
        outputs=output_json,
    )
    
    gr.Examples(
        examples=[[None, "look at the red cube"]],
        inputs=[image_input, task_input],
        run_on_click=False,
    )

# Launch dengan share=True untuk public URL
print("\n" + "="*70)
print("Launching Gradio Public Backend...")
print("="*70)

demo.launch(
    share=True,  # Generate public Gradio link
    show_error=True,
    quiet=False,
)
